# Phase 3: Data Preprocessing & Feature Engineering

This notebook prepares the **IBM Telco Customer Churn** dataset for machine
learning. The analysis steps live in `01_exploratory_data_analysis.ipynb`.

**Objectives**
- Remove unnecessary / leakage columns
- Fix incorrect data types and handle missing values
- Engineer new features
- Encode categorical features and scale numerical ones
- Split into train/test and persist a reusable preprocessing pipeline

**Data note:** 11 customers have a blank `Total Charges` (all with
`Tenure Months` = 0). They are imputed with 0 here, so all 7,043 rows are kept.

In [1]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

DATA_RAW = "../data/raw"
DATA_PROCESSED = "../data/processed"
MODELS_DIR = "../models"

RAW_FILE = os.path.join(DATA_RAW, "Telco_customer_churn.xlsx")
CLEANED_FILE = os.path.join(DATA_PROCESSED, "cleaned_telco_customer_churn.csv")
X_TRAIN_RAW = os.path.join(DATA_PROCESSED, "X_train_raw.csv")
X_TEST_RAW = os.path.join(DATA_PROCESSED, "X_test_raw.csv")
X_TRAIN_CSV = os.path.join(DATA_PROCESSED, "X_train.csv")
X_TEST_CSV = os.path.join(DATA_PROCESSED, "X_test.csv")
Y_TRAIN_CSV = os.path.join(DATA_PROCESSED, "y_train.csv")
Y_TEST_CSV = os.path.join(DATA_PROCESSED, "y_test.csv")
PREPROCESSOR_PKL = os.path.join(MODELS_DIR, "preprocessor.pkl")

# Load Data

In [2]:
df = pd.read_excel(RAW_FILE)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

In [4]:
print(df.shape)

(7043, 33)


In [5]:
df.columns.tolist()

['CustomerID',
 'Count',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'CLTV',
 'Churn Reason']

In [6]:
df.isnull().sum()

CustomerID              0
Count                   0
Country                 0
State                   0
City                    0
Zip Code                0
Lat Long                0
Latitude                0
Longitude               0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Label             0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         5174
dtype: int64

In [7]:
df.dtypes

CustomerID            object
Count                  int64
Country               object
State                 object
City                  object
Zip Code               int64
Lat Long              object
Latitude             float64
Longitude            float64
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service         object
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Streaming TV          object
Streaming Movies      object
Contract              object
Paperless Billing     object
Payment Method        object
Monthly Charges      float64
Total Charges         object
Churn Label           object
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason          object
dtype: object

In [8]:
df_clean = df.copy()

In [9]:
print(df.shape)

print(df_clean.shape)

(7043, 33)
(7043, 33)


# Drop Unnecessary Columns

In [10]:
columns_to_drop = [
    "CustomerID",
    "Count",
    "Country",
    "State",
    "Lat Long",
    "Churn Score",
    "Churn Reason"
]

In [11]:
columns_to_drop

['CustomerID',
 'Count',
 'Country',
 'State',
 'Lat Long',
 'Churn Score',
 'Churn Reason']

In [12]:
df_clean.drop(columns=columns_to_drop, inplace=True)

In [13]:
df_clean.shape

(7043, 26)

In [14]:
df_clean.columns.tolist()

['City',
 'Zip Code',
 'Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'CLTV']

In [15]:
removed = sorted(set(df.columns) - set(df_clean.columns))
removed

['Churn Reason',
 'Churn Score',
 'Count',
 'Country',
 'CustomerID',
 'Lat Long',
 'State']

In [16]:
missing = df_clean.isnull().sum()

missing

City                 0
Zip Code             0
Latitude             0
Longitude            0
Gender               0
Senior Citizen       0
Partner              0
Dependents           0
Tenure Months        0
Phone Service        0
Multiple Lines       0
Internet Service     0
Online Security      0
Online Backup        0
Device Protection    0
Tech Support         0
Streaming TV         0
Streaming Movies     0
Contract             0
Paperless Billing    0
Payment Method       0
Monthly Charges      0
Total Charges        0
Churn Label          0
Churn Value          0
CLTV                 0
dtype: int64

In [17]:
missing[missing > 0]

Series([], dtype: int64)

# Missing Values in Total Charges

In [18]:
df_clean["Total Charges"].head(10)

0     108.15
1     151.65
2      820.5
3    3046.05
4     5036.3
5     528.35
6      39.65
7      20.15
8    4749.15
9       30.2
Name: Total Charges, dtype: object

In [19]:
df_clean["Total Charges"].dtype

dtype('O')

In [20]:
df_clean["Total Charges"].apply(type).value_counts()

Total Charges
<class 'float'>    6708
<class 'int'>       324
<class 'str'>        11
Name: count, dtype: int64

In [21]:
(df_clean["Total Charges"] == "").sum()

np.int64(0)

In [22]:
(df_clean["Total Charges"].str.strip() == "").sum()

np.int64(11)

In [23]:
df_clean["Total Charges"] = df_clean["Total Charges"].replace(" ", np.nan)

In [24]:
df_clean["Total Charges"].isnull().sum()

np.int64(11)

In [25]:
df_clean["Total Charges"] = pd.to_numeric(
    df_clean["Total Charges"],
    errors="coerce"
)

In [26]:
df_clean["Total Charges"].dtype

dtype('float64')

In [27]:
df_clean[df_clean["Total Charges"].isnull()]

,City,Zip Code,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,...,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,CLTV
2234,San Bernardino,92408,34.084909,-117.258107,Female,No,Yes,No,0,No,...,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No,0,2578
2438,Independence,93526,36.869584,-118.189241,Male,No,No,No,0,Yes,...,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No,0,5504
2568,San Mateo,94401,37.590421,-122.306467,Female,No,Yes,No,0,Yes,...,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No,0,2048
2667,Cupertino,95014,37.306612,-122.080621,Male,No,Yes,Yes,0,Yes,...,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No,0,4950
2856,Redcrest,95569,40.363446,-123.835041,Female,No,Yes,No,0,No,...,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No,0,4740
4331,Los Angeles,90029,34.089953,-118.294824,Male,No,Yes,Yes,0,Yes,...,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No,0,2019
4687,Sun City,92585,33.739412,-117.173334,Male,No,Yes,Yes,0,Yes,...,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No,0,2299
5104,Ben Lomond,95005,37.078873,-122.090386,Female,No,Yes,Yes,0,Yes,...,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No,0,3763
5719,La Verne,91750,34.144703,-117.770299,Male,No,Yes,Yes,0,Yes,...,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No,0,4890
6772,Bell,90201,33.970343,-118.171368,Female,No,Yes,Yes,0,Yes,...,Yes,No,Two year,No,Mailed check,73.35,NaN,No,0,2342


In [28]:
df_clean["Total Charges"] = df_clean["Total Charges"].fillna(0)

In [29]:
df_clean.isnull().sum().sum()

np.int64(0)

The blank `Total Charges` rows are all customers with
`Tenure Months` = 0 who have not yet accumulated charges, so imputing 0 is
appropriate.

In [30]:
print(df_clean.shape)

(7043, 26)


In [31]:
df_clean.dtypes

City                  object
Zip Code               int64
Latitude             float64
Longitude            float64
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service         object
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Streaming TV          object
Streaming Movies      object
Contract              object
Paperless Billing     object
Payment Method        object
Monthly Charges      float64
Total Charges        float64
Churn Label           object
Churn Value            int64
CLTV                   int64
dtype: object

In [32]:
df_clean.isnull().sum()

City                 0
Zip Code             0
Latitude             0
Longitude            0
Gender               0
Senior Citizen       0
Partner              0
Dependents           0
Tenure Months        0
Phone Service        0
Multiple Lines       0
Internet Service     0
Online Security      0
Online Backup        0
Device Protection    0
Tech Support         0
Streaming TV         0
Streaming Movies     0
Contract             0
Paperless Billing    0
Payment Method       0
Monthly Charges      0
Total Charges        0
Churn Label          0
Churn Value          0
CLTV                 0
dtype: int64

In [33]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   City               7043 non-null   object 
 1   Zip Code           7043 non-null   int64  
 2   Latitude           7043 non-null   float64
 3   Longitude          7043 non-null   float64
 4   Gender             7043 non-null   object 
 5   Senior Citizen     7043 non-null   object 
 6   Partner            7043 non-null   object 
 7   Dependents         7043 non-null   object 
 8   Tenure Months      7043 non-null   int64  
 9   Phone Service      7043 non-null   object 
 10  Multiple Lines     7043 non-null   object 
 11  Internet Service   7043 non-null   object 
 12  Online Security    7043 non-null   object 
 13  Online Backup      7043 non-null   object 
 14  Device Protection  7043 non-null   object 
 15  Tech Support       7043 non-null   object 
 16  Streaming TV       7043 

In [34]:
print("=" * 50)
print("DATASET VALIDATION")
print("=" * 50)

print(f"Rows           : {df_clean.shape[0]}")
print(f"Columns        : {df_clean.shape[1]}")
print(f"Missing Values : {df_clean.isnull().sum().sum()}")
print(f"Duplicate Rows : {df_clean.duplicated().sum()}")

print("\nData Types\n")
print(df_clean.dtypes.value_counts())

DATASET VALIDATION
Rows           : 7043
Columns        : 26
Missing Values : 0
Duplicate Rows : 0

Data Types

object     18
int64       4
float64     4
Name: count, dtype: int64


# Feature Engineering

In [35]:
target = "Churn Label"

numerical_features = [
    "Tenure Months",
    "Monthly Charges",
    "Total Charges"
]

categorical_features = [
    "Gender",
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Paperless Billing",
    "Payment Method"
]

In [36]:
print("Numerical Features   :", len(numerical_features))
print("Categorical Features :", len(categorical_features))

Numerical Features   : 3
Categorical Features : 16


## Derived Features

In [37]:
df_clean["Avg Monthly Spend"] = np.where(
    df_clean["Tenure Months"] == 0,
    0,
    df_clean["Total Charges"] / df_clean["Tenure Months"]
)

In [38]:
df_clean["Customer Age Group"] = pd.cut(
    df_clean["Tenure Months"],
    bins=[-1, 12, 24, 48, 72],
    labels=["New", "Growing", "Loyal", "Very Loyal"]
)

In [39]:
df_clean[[
    "Tenure Months",
    "Total Charges",
    "Avg Monthly Spend",
    "Customer Age Group"
]].head()

,Tenure Months,Total Charges,Avg Monthly Spend,Customer Age Group
0,2,108.15,54.075000,New
1,2,151.65,75.825000,New
2,8,820.50,102.562500,New
3,28,3046.05,108.787500,Loyal
4,49,5036.30,102.781633,Very Loyal


In [40]:
if "Avg Monthly Spend" not in numerical_features:
    numerical_features.append("Avg Monthly Spend")

if "Customer Age Group" not in categorical_features:
    categorical_features.append("Customer Age Group")

# Feature Encoding

In [41]:
pd.crosstab(
    df_clean["Churn Label"],
    df_clean["Churn Value"]
)

Churn Value,0,1
Churn Label,,
No,5174,0
Yes,0,1869


## Target Handling

`Churn Value` is simply the numeric representation of `Churn Label`, so it is
removed to avoid duplicate target information. `City`, `Zip Code`, `Latitude`,
`Longitude`, and `CLTV` are not used as model features and are excluded from
`X` as well.

In [42]:
non_model_columns = [
    "City",
    "Zip Code",
    "Latitude",
    "Longitude",
    "CLTV"
]

X = df_clean.drop(columns=[target, "Churn Value"] + non_model_columns)
y = df_clean[target]

In [43]:
y = df_clean[target].map({"Yes": 1, "No": 0})

In [44]:
y.value_counts()

Churn Label
0    5174
1    1869
Name: count, dtype: int64

In [45]:
df_clean.to_csv(CLEANED_FILE, index=False)

# Train / Test Split

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [47]:
print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Training Shape : (5634, 21)
Testing Shape  : (1409, 21)
Churn Label
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Churn Label
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [48]:
X_train.to_csv(X_TRAIN_RAW, index=False)
X_test.to_csv(X_TEST_RAW, index=False)

# Build Preprocessing Pipeline

## Numeric & Categorical Pipelines

In [49]:
# No missing values remain, but the imputer keeps the pipeline
# defensive if new data arrives with missing entries.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [50]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [51]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

In [52]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

In [53]:
print(X_train_processed.shape)

print(X_test_processed.shape)

(5634, 51)
(1409, 51)


# Save Pipeline

In [54]:
joblib.dump(preprocessor, PREPROCESSOR_PKL)

['../models\\preprocessor.pkl']

# Save Processed Dataset

In [55]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

In [56]:
X_train_processed.to_csv(X_TRAIN_CSV, index=False)
X_test_processed.to_csv(X_TEST_CSV, index=False)

y_train.to_csv(Y_TRAIN_CSV, index=False)
y_test.to_csv(Y_TEST_CSV, index=False)

In [57]:
files = [
    X_TRAIN_CSV,
    X_TEST_CSV,
    Y_TRAIN_CSV,
    Y_TEST_CSV,
    PREPROCESSOR_PKL
]

for file in files:
    print(file, ":", os.path.exists(file))

../data/processed\X_train.csv : True
../data/processed\X_test.csv : True
../data/processed\y_train.csv : True
../data/processed\y_test.csv : True
../models\preprocessor.pkl : True


# Final Validation

In [58]:
summary = {
    "Original Rows": df.shape[0],
    "Original Columns": df.shape[1],
    "Final Rows": df_clean.shape[0],
    "Final Columns": df_clean.shape[1],
    "Missing Values": int(df_clean.isnull().sum().sum()),
    "Duplicate Rows": int(df_clean.duplicated().sum()),
    "Numerical Features": len(numerical_features),
    "Categorical Features": len(categorical_features),
}

pd.DataFrame(summary.items(), columns=["Metric", "Value"])

,Metric,Value
0,Original Rows,7043
1,Original Columns,33
2,Final Rows,7043
3,Final Columns,28
4,Missing Values,0
5,Duplicate Rows,0
6,Numerical Features,4
7,Categorical Features,17


In [59]:
print("=" * 60)
print("PREPROCESSING COMPLETED")
print("=" * 60)

print(f"Training Samples  : {X_train.shape[0]}")
print(f"Testing Samples   : {X_test.shape[0]}")

print(f"Original Features  : {X.shape[1]}")
print(f"Processed Features : {X_train_processed.shape[1]}")

print("\nMissing Values")
print(X_train_processed.isnull().sum().sum())

print("\nPipeline Saved")
print(os.path.exists(PREPROCESSOR_PKL))

PREPROCESSING COMPLETED
Training Samples  : 5634
Testing Samples   : 1409
Original Features  : 21
Processed Features : 51

Missing Values
0

Pipeline Saved
True
